<a href="https://colab.research.google.com/github/dechl-98/ChavezDavid2534532021/blob/main/ClaveG/An%C3%A1lisis_de_Asociaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#DEsarrollo parcial 4, Empezamos con la data que corresponde a ASOCIACION
#Importando las librerías que se necesitarán para trabajar y presentar los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [2]:
#Definimos una variable url para mandar a llamar nuestro csv desde el repositorio de github
url = "https://raw.githubusercontent.com/dechl-98/ChavezDavid2534532021/refs/heads/main/ClaveG/clave_G_asociacion.csv"
df = pd.read_csv(url)
df.head()

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,G-T0001,G-C0072,2026-02-20,Construccion,Arena,1,Tienda
1,G-T0001,G-C0072,2026-02-20,Herramientas,Destornillador,1,Tienda
2,G-T0001,G-C0072,2026-02-20,Pintura,Lija,3,Tienda
3,G-T0002,G-C0057,2026-01-26,Electricidad,Cable,2,App
4,G-T0002,G-C0057,2026-01-26,Electricidad,Interruptor,1,App


Las primeras filas del data set nos muestra una tabla en la cual va enfocada a ferretería. Está formada por siete campos que son: transaccion_id, cliente_id, fecha, categoría, item, cantidad y canal. Los tipos de datos son los siguientes: transaccion_id, cliente_id, categoría, item y canal son de tipo varchar otexto, el campo fecha corresponde a un campo de tipo fecha y el campo cantidad tiene un tipeo de int o entero (número).

In [3]:
#Verificamos valores nulos, duplicados y tipos de datos.
#verificamos la información general del dataset
df.info()
# verificamos valores nulos por columna y los registros que podrían ser duplicados
print("Valores nulos por columna:")
print(df.isnull().sum())
# Registros duplicados
print("\nCantidad de registros duplicados:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622 entries, 0 to 621
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  622 non-null    object
 1   cliente_id      622 non-null    object
 2   fecha           622 non-null    object
 3   categoria       622 non-null    object
 4   item            622 non-null    object
 5   cantidad        622 non-null    int64 
 6   canal           621 non-null    object
dtypes: int64(1), object(6)
memory usage: 34.1+ KB
Valores nulos por columna:
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64

Cantidad de registros duplicados: 1


Hay un total de siete columnas y de 622 registros, de todas las columnas encontramos que hay un solo registro duplicado y corresponde al campo canal. Analizando eso vemos que no infiere en la estadística ni en la data necesaria.

In [4]:
#Revisamos valores atípicos
variables_numericas = ["cantidad"]
for columna in variables_numericas:
    media = df[columna].mean()
    desviacion = df[columna].std()
    limite_inferior = media - 2 * desviacion
    limite_superior = media + 2 * desviacion
    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    print(columna, limite_inferior, limite_superior, len(outliers))


cantidad -0.08644112683185035 3.0028398406582166 19


In [5]:
#Generamos estadística descriptiva
df.describe

,cantidad
count,622.000000
mean,1.458199
std,0.772320
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,4.000000


In [8]:
# 4. Selección de columnas de asociación
columnas_asociacion = [
    "categoria", "item", "cantidad", "canal"
]
df_asociacion = df[columnas_asociacion].copy()
df_asociacion_bool = df_asociacion.astype(bool)


En la sección anterior se han tomado los casos en los cuales se puede hacer una asociación para anlizar datos, catgorias, item, cantidad

In [9]:
#Buscamos patrones frecuentes en la asociación antes establecida y las reglas
from itertools import combinations
reglas = []
n = len(df_asociacion_bool)

# Iteramos diferentes tamaños
for k in range(2, len(columnas_asociacion) + 1):
    for itemset in combinations(columnas_asociacion, k):
        itemset_list = list(itemset)
        # Calculando
        soporte_itemset = df_asociacion_bool[itemset_list].all(axis=1).mean()

        # Minimo apoyo
        if soporte_itemset < 0.10:
            continue

        # Generando asociaciones potenciales
        for tam_antecedente in range(1, k):
            for antecedente in combinations(itemset, tam_antecedente):
                antecedente_set = set(antecedente)
                consecuente_set = set(itemset) - antecedente_set

                # Calculando apoyos para antecendentes y consecuentes
                soporte_antecedente = df_asociacion_bool[list(antecedente_set)].all(axis=1).mean()
                soporte_consecuente = df_asociacion_bool[list(consecuente_set)].all(axis=1).mean()


                confianza = soporte_itemset / soporte_antecedente
                lift = confianza / soporte_consecuente

                # Minimos
                if confianza >= 0.50:
                    reglas.append({
                        "antecedente": ", ".join(sorted(antecedente_set)),
                        "consecuente": ", ".join(sorted(consecuente_set)),
                        "soporte": soporte_itemset,
                        "confianza": confianza,
                        "lift": lift
                    })

# Creando el dataframe con los resultados
reglas_df = pd.DataFrame(reglas).sort_values(
    by=["confianza", "soporte"], ascending=False
)

if not reglas_df.empty:
    display(reglas_df)
else:
    print("No se encontraron reglas que cumplan con los umbrales establecidos.")

,antecedente,consecuente,soporte,confianza,lift
0,categoria,item,1.0,1.0,1.0
1,item,categoria,1.0,1.0,1.0
2,categoria,cantidad,1.0,1.0,1.0
3,cantidad,categoria,1.0,1.0,1.0
4,categoria,canal,1.0,1.0,1.0
5,canal,categoria,1.0,1.0,1.0
6,item,cantidad,1.0,1.0,1.0
7,cantidad,item,1.0,1.0,1.0
8,item,canal,1.0,1.0,1.0
9,canal,item,1.0,1.0,1.0


los patrones frecuentes son canal, cantidad y categoría; cantidad, categoría e ítem; canal,categoría e ítem. Ello demuestra los métodos mas utilizadfos para la compra

In [10]:
# 6. Guardamos los resultados obtenidos del dataframe
df_asociacion_bool.to_csv("https://raw.githubusercontent.com/dechl-98/ChavezDavid2534532021/refs/heads/main/ClaveG/clave_G_asociacion.csv", index=False)
reglas_df.to_csv("https://raw.githubusercontent.com/dechl-98/ChavezDavid2534532021/refs/heads/main/ClaveG/clave_G_asociacion.csv", index=False)
